## Autoregressive Generation : with vs without KV cache

### 1- Introduction


##### Objectives : 
- understand concretly how KV cache accelerate token generation for a given Transformer decoder model.
- Make benchmarks to measure the impact of having a KV cache.

##### Main ideas : 
- Without KV cache: for each iteration the transformer recalculate the attention for all the sequence for every new generated oken
- With KV cache :  the Keys and Values of tokens already calculated are memorized 

##### Plan : 
- will  be done 

##### Choosen model for exepriment : 
- GPT 2-small (124M param ) : a tiny small language model but with a simple and typical transformer decoder.

### 2- Imports

In [17]:
import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Devide available : {device}")

dtype = torch.float32

Devide available : cuda


In [18]:
# loading tokenizer 
tokenizer=GPT2TokenizerFast.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2").to(device).to(dtype)


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 2141.91it/s]


In [24]:
# deactivate the dropout for better benchmark results 
model.eval()

#model info
n_params = sum(p.numel() for p in model.parameters())
print(f"GPT-2 small : {n_params/1e6:.1f}M parameters ")
print(f"Layers (blocs Transformer) : {model.config.n_layer}")
print(f"Attention head for each layer : {model.config.n_head}")
print(f"Hidden dim : {model.config.n_embd}")

GPT-2 small : 124.4M parameters 
Layers (blocs Transformer) : 12
Attention head for each layer : 12
Hidden dim : 768


### 3- Theory 

In a decode language model,  the decoder predict the token from the previous tokens. This generation is being done token by token.

- prompt -> predict token n°1
- prompt + token n°2  -> predict token n°2
- prompt + token 1 + token 2 - > predict token n°3 

....

In each step , the number of tokens in the input model is getting bigger generating some performance issue 